In [1]:
from wikifin_rag.db_client import PostgresClient

In [2]:
db_client = PostgresClient()

In [3]:
db_client.open_connection()

db_client.cur.execute(
    """
    SELECT
        c.document_id || '_' || c.chunk_id AS id,
        d.title,
        d.section,
        c.content
    FROM chunks c
    JOIN documents d
    ON c.document_id = d.id
    WHERE language = 'nl'
    LIMIT 100;
    """
)
documents = db_client.cur.fetchall()

db_client.close_connection()

In [4]:
doc = documents[0]

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate either a university student or a young professional.
Formulate 3 questions this student/professional might ask based on an article excerpt. The excerpt
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI()

In [8]:
import json
user_prompt = json.dumps(doc)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

In [13]:
result.questions

['Welke kosten en taksen moet je betalen als je in een tak 23-levensverzekering belegt?',
 'Waarom is het belangrijk om de kosten van een tak 23-levensverzekering goed te kennen voordat je erin belegt?',
 'Hoe kunnen de kosten van een tak 23-levensverzekering je opbrengst beïnvloeden?']

In [19]:
from wikifin_rag.evaluation_utils import llm_structured_retry, map_progress, calc_total_price
from concurrent.futures import ThreadPoolExecutor

In [15]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [16]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

In [20]:
calc_total_price(usages)

0.052830750000000024

In [21]:
from wikifin_rag.config import PROJECT_ROOT

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

dest = PROJECT_ROOT / "data"
dest.mkdir(parents=True, exist_ok=True)

df_ground_truth.to_csv(PROJECT_ROOT / "data/ground_truth.csv", index=False)

OSError: Cannot save file into a non-existent directory: '/Users/bastienwinant/Desktop/projects/wikifin-rag/data'